# Medición de tiempo, CPU y memoria del procesamiento

Este notebook mide cuánto tarda y cuántos recursos usa cada etapa del procesamiento del dataset de BCS, con los datos reales del bucket de S3 y en una máquina de Colab. Sirve para justificar con números si el proyecto necesita herramientas de Big Data.

Las etapas son las mismas del notebook de preparación.

1. Descarga de los datos crudos desde S3.
2. Inventario de las fotos con su huella SHA-256.
3. Conversión a JPG de las 580 fotos del dataset.
4. Integración (índice, etiqueta de BCS y pliegues) con pandas y con PySpark, 5 veces cada una.
5. La misma integración con un volumen simulado de 10 y 100 veces los datos, para ver cómo crece el tiempo.

De cada etapa se registra el tiempo, el tiempo de CPU (cpu_pct = 100 equivale a un núcleo ocupado todo el tiempo) y la memoria máxima, que en PySpark incluye la de la JVM. La integración también verifica que pandas y PySpark generen exactamente las mismas tablas y que esas tablas sean iguales al dataset del bucket.

Al final se descarga `tiempos_recursos.csv` y `entorno.json` para la carpeta `reports/bigdata/resultados/` del repositorio. Los datos se quedan en Colab.

## Configuración

Las llaves de AWS y el nombre del bucket se leen de los secretos de Colab `AWS_KEY_ID`, `AWS_SECRET` y `S3_BUCKET`. Los scripts de la integración se bajan del repositorio del proyecto (`src/bigdata/pyspark/`). PySpark necesita Java, y si la máquina de Colab no lo tiene, la celda lo instala.

Para que la medición sea comparable, conviene correr el notebook completo en una sesión nueva (Entorno de ejecución, Desconectar y borrar el entorno de ejecución) y sin otras pestañas de Colab trabajando.

In [ ]:
# Parámetros
PREFIJO_FOTOS = "data/fotosextraidas/"
KEY_VISTAS = "bcs/intermedios/vistas.csv"
PREFIJO_DATASET = "bcs/"
REPO = "https://github.com/AIPorDefinir/ai-model-tutsipink.git"
TRABAJO = "/content/medicion"
REPETICIONES = 5            # corridas de la integración con los datos reales
ESCALAS = [10, 100]         # volumen simulado de la integración (veces los datos reales)
REPETICIONES_ESCALA = 3
HILOS = 4                   # fotos que se convierten a la vez, igual que en la preparación
CALIDAD = 95

In [ ]:
!pip install -q boto3 pillow-heif psutil pyspark==3.5.3
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path


def version_java():
    if shutil.which("java") is None:
        return None, ""
    linea = next((l for l in subprocess.run(["java", "-version"], capture_output=True, text=True).stderr.splitlines()
                  if "version" in l), "")
    m = re.search(r'version "(\d+)(?:\.(\d+))?', linea)
    mayor = int(m.group(1)) if m else None
    return (int(m.group(2)) if mayor == 1 else mayor), linea  # "1.8" es Java 8


# PySpark 3.5 funciona con Java 8, 11 o 17; si no hay ninguna de esas, se instala Java 17
mayor, linea = version_java()
if mayor not in (8, 11, 17):
    subprocess.run(["apt-get", "-qq", "update"], check=True)
    subprocess.run(["apt-get", "-qq", "install", "-y", "openjdk-17-jre-headless"], check=True)
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]
    mayor, linea = version_java()
print("Java:", linea)

trabajo = Path(TRABAJO)
trabajo.mkdir(parents=True, exist_ok=True)
repo = trabajo / "ai-model-tutsipink"
SCRIPTS = repo / "src" / "bigdata" / "pyspark"
if not SCRIPTS.exists():
    # Los scripts están en main; mientras no se fusione el PR #98 solo están en chore/unificar-repos
    for rama in ("main", "chore/unificar-repos"):
        shutil.rmtree(repo, ignore_errors=True)
        subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", rama, REPO, str(repo)], check=True)
        if SCRIPTS.exists():
            print("Scripts de la rama", rama)
            break
assert SCRIPTS.exists(), "No encontré src/bigdata/pyspark/ en main ni en chore/unificar-repos"
sys.path.insert(0, str(SCRIPTS))
from medicion import Medidor, entorno  # noqa: E402

print("Máquina:", entorno())

In [ ]:
!pip install -q boto3
import boto3
from botocore.exceptions import ClientError
from pathlib import Path
from google.colab import userdata

BUCKET = userdata.get("S3_BUCKET")
s3 = boto3.client("s3", region_name="mx-central-1",
                  aws_access_key_id=userdata.get("AWS_KEY_ID"),
                  aws_secret_access_key=userdata.get("AWS_SECRET"))


def listar(prefijo):
    # Llaves de los archivos del bucket bajo un prefijo
    for pagina in s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET, Prefix=prefijo):
        for o in pagina.get("Contents", []):
            if not o["Key"].endswith("/"):
                yield o["Key"]


def bajar_dataset(destino):
    # Baja las tres tablas del dataset; si no están en PREFIJO_DATASET, dice dónde sí hay un índice
    destino = Path(destino)
    destino.mkdir(parents=True, exist_ok=True)
    prefijo = PREFIJO_DATASET.strip("/") + "/"
    for nombre in ("indice_imagenes.csv", "vacas.csv", "evaluaciones.csv"):
        try:
            s3.download_file(BUCKET, prefijo + nombre, str(destino / nombre))
        except ClientError:
            otros = [k for k in listar("") if k.endswith("indice_imagenes.csv")]
            raise FileNotFoundError(f"No encontré {prefijo + nombre} en el bucket. Índices que sí existen: {otros}. "
                                    "Corrige PREFIJO_DATASET y vuelve a correr esta celda.")
    return destino


registros = []


def registrar(etapa, implementacion, resultado, **extra):
    registros.append({"etapa": etapa, "implementacion": implementacion, **resultado, **extra})
    print(f"{etapa} | {implementacion} | {resultado} | {extra}")

## 1. Descarga de los datos crudos desde S3

Se bajan todas las fotos de `PREFIJO_FOTOS`, el `vistas.csv` y las tablas del dataset. La descarga es un archivo a la vez, igual que en los notebooks de comprensión y preparación.

In [ ]:
DATOS = trabajo / "datos"
DIR_FOTOS = DATOS / "fotosextraidas"
DIR_FOTOS.mkdir(parents=True, exist_ok=True)
llaves = list(listar(PREFIJO_FOTOS))
with Medidor() as m:
    for k in llaves:
        archivo = DIR_FOTOS / k[len(PREFIJO_FOTOS):]
        archivo.parent.mkdir(parents=True, exist_ok=True)
        s3.download_file(BUCKET, k, str(archivo))
    s3.download_file(BUCKET, KEY_VISTAS, str(DATOS / "vistas.csv"))
    DATASET = bajar_dataset(DATOS / "dataset_bcs")
gb = sum(p.stat().st_size for p in DIR_FOTOS.rglob("*") if p.is_file()) / 1e9
registrar("1. descarga de S3", "boto3", m.resultado, repeticion=1, escala=1, archivos=len(llaves), gb=round(gb, 2))

## 2. Inventario y huellas

Se recorren las carpetas de fotos y se calcula la huella SHA-256 de cada archivo, que es lo que usa la preparación para detectar copias exactas.

In [ ]:
import hashlib


def sha256(ruta, bloque=1 << 20):
    h = hashlib.sha256()
    with open(ruta, "rb") as f:
        for b in iter(lambda: f.read(bloque), b""):
            h.update(b)
    return h.hexdigest()


EXT_FOTO = {".heic", ".heif", ".jpg", ".jpeg", ".png"}
archivos = sorted(p for p in DIR_FOTOS.rglob("*")
                  if p.is_file() and p.suffix.lower() in EXT_FOTO and not p.name.startswith("."))
with Medidor() as m:
    huellas = [sha256(p) for p in archivos]
registrar("2. inventario y huellas", "Python", m.resultado, repeticion=1, escala=1, archivos=len(archivos),
          gb=round(sum(p.stat().st_size for p in archivos) / 1e9, 2))
print(f"Archivos: {len(archivos)} | huellas distintas: {len(set(huellas))}")

## 3. Conversión a JPG

Se convierten las mismas fotos que convierte la preparación, que son las 580 del índice (las excluidas no se convierten). Las reglas son las mismas: los HEIC pasan a JPG con calidad 95, los JPG con orientación EXIF distinta de 1 se enderezan y los JPG que ya venían derechos se copian sin cambios. Se procesan `HILOS` fotos a la vez. Al final se compara cuántas se convirtieron y cuántas se copiaron con lo que reportó la preparación.

In [ ]:
import os
from concurrent.futures import ThreadPoolExecutor

import pandas as pd
import pillow_heif
from PIL import Image, ImageOps

pillow_heif.register_heif_opener()
JPG = trabajo / "jpg"


def convertir(origen, destino):
    origen, destino = Path(origen), Path(destino)
    destino.parent.mkdir(parents=True, exist_ok=True)
    ext = origen.suffix.lower()
    try:
        with Image.open(origen) as im:
            exif = im.getexif()
            girar = ext not in {".heic", ".heif"} and exif.get(0x0112, 1) not in (None, 1)
            tmp = destino.with_suffix(".tmp")
            if ext in {".jpg", ".jpeg"} and not girar:
                shutil.copy2(origen, tmp)
                estado = "copiada"
            else:
                final = ImageOps.exif_transpose(im) if girar else im
                exif[0x0112] = 1
                extra = {"quality": CALIDAD, "exif": exif.tobytes()}
                if im.info.get("icc_profile"):
                    extra["icc_profile"] = im.info["icc_profile"]
                final.convert("RGB").save(tmp, "JPEG", **extra)
                estado = "convertida"
            os.replace(tmp, destino)
            return estado
    except Exception as e:  # noqa: BLE001
        return f"error: {e}"[:120]


idx = pd.read_csv(DATASET / "indice_imagenes.csv", dtype=str, keep_default_na=False)
tareas = [(DIR_FOTOS / f.archivo_original, JPG / f.archivo_jpg) for f in idx.itertuples()]
if JPG.exists():
    shutil.rmtree(JPG)
with Medidor() as m:
    with ThreadPoolExecutor(max_workers=HILOS) as ex:
        estados = list(ex.map(lambda t: convertir(*t), tareas))
gb = sum(p.stat().st_size for p in JPG.rglob("*.jpg")) / 1e9
registrar("3. conversión a JPG", f"Pillow, {HILOS} hilos", m.resultado, repeticion=1, escala=1,
          archivos=len(tareas), gb=round(gb, 2))
display(pd.Series(estados).value_counts().to_frame("fotos"))

## 4. Integración con pandas y con PySpark

La integración une las fotos, las vistas y las evaluaciones, calcula la etiqueta de BCS de cada vaca, asigna los pliegues y escribe las tres tablas del dataset. Sus entradas (`fotos.csv`, `bitacora.csv` y `vistas.csv`) las genera la preparación en su carpeta de trabajo de Colab, que no se guarda en el bucket, así que `preparar_entradas.py` las reconstruye a partir de las tablas del dataset y del `vistas.csv` del bucket.

`comparar_tiempos.py` corre `indice_pandas.py` e `indice_spark.py` (PySpark en modo local, con todos los núcleos de la máquina) `REPETICIONES` veces cada uno, cada corrida en un proceso nuevo. Mide cada corrida completa, incluido el arranque de Spark, y al final verifica que las dos versiones den las mismas tablas y que sean iguales al dataset del bucket.

In [ ]:
def correr(args, puede_fallar=False):
    r = subprocess.run([sys.executable, *map(str, args)], cwd=SCRIPTS, capture_output=True, text=True)
    print(r.stdout[-5000:])
    if r.returncode != 0:
        print(r.stderr[-5000:])
        if not puede_fallar:
            raise RuntimeError(f"Falló {args[0]}")
        print(f"ATENCIÓN: {args[0]} terminó con diferencias. Revisa la verificación de arriba.")


def registrar_integracion(carpeta, escala):
    t = pd.read_csv(carpeta / "tiempos.csv")
    filas = len(pd.read_csv(carpeta / "pandas" / "indice_imagenes.csv"))
    for f in t.itertuples():
        registrar("4. integración", f.implementacion,
                  {"segundos": f.t_total_proceso, "cpu_s": f.cpu_s, "cpu_pct": f.cpu_pct, "mem_max_mb": f.mem_max_mb},
                  repeticion=f.repeticion, escala=escala, archivos=filas, t_sesion=f.t_sesion,
                  t_procesamiento=f.t_procesamiento)


ENTRADA = trabajo / "entrada"
correr(["preparar_entradas.py", "--dataset", DATASET, "--vistas", DATOS / "vistas.csv", "--salida", ENTRADA])
correr(["comparar_tiempos.py", "--entrada", ENTRADA, "--salida", trabajo / "integracion",
        "--referencia", DATASET, "--repeticiones", REPETICIONES], puede_fallar=True)
registrar_integracion(trabajo / "integracion", 1)

## 5. Integración con volumen simulado

Para ver a partir de qué tamaño cambiaría la conclusión, la misma integración se corre con entradas 10 y 100 veces más grandes. `preparar_entradas.py --multiplicar N` repite todas las vacas N veces con IDs distintos, así que es un volumen simulado a partir de los datos reales y no son vacas nuevas. Aquí no se compara contra el bucket, porque el dataset real no tiene esas copias, pero sí se verifica que pandas y PySpark coincidan.

In [ ]:
for n in ESCALAS:
    entrada_n, salida_n = trabajo / f"entrada_x{n}", trabajo / f"integracion_x{n}"
    correr(["preparar_entradas.py", "--dataset", DATASET, "--vistas", DATOS / "vistas.csv", "--salida", entrada_n,
            "--multiplicar", n])
    correr(["comparar_tiempos.py", "--entrada", entrada_n, "--salida", salida_n,
            "--repeticiones", REPETICIONES_ESCALA], puede_fallar=True)
    registrar_integracion(salida_n, n)

## Resumen y archivos de resultados

La tabla resume cada etapa con la mediana de sus corridas. `tiempos_recursos.csv` tiene todas las corridas y `entorno.json` describe la máquina y las versiones. Los dos se descargan para la carpeta `reports/bigdata/resultados/` del repositorio.

In [ ]:
import json
import platform

import pyspark
from google.colab import files

tabla = pd.DataFrame(registros)
cols = ["etapa", "implementacion", "escala", "repeticion", "archivos", "gb", "segundos", "cpu_s", "cpu_pct",
        "mem_max_mb", "t_sesion", "t_procesamiento"]
tabla = tabla.reindex(columns=cols)
resumen = (tabla.groupby(["etapa", "implementacion", "escala"], sort=False)
           [["archivos", "segundos", "cpu_pct", "mem_max_mb", "t_sesion", "t_procesamiento"]].median().round(3))
display(resumen)

carpeta = Path("/content/resultados")
carpeta.mkdir(exist_ok=True)
tabla.to_csv(carpeta / "tiempos_recursos.csv", index=False)
java = next(l for l in subprocess.run(["java", "-version"], capture_output=True, text=True).stderr.splitlines() if "version" in l)
info = {**entorno(), "python": platform.python_version(), "pandas": pd.__version__, "pyspark": pyspark.__version__,
        "java": java, "maquina": "Google Colab", "fecha_utc": pd.Timestamp.now(tz="UTC").strftime("%Y-%m-%d %H:%M")}
(carpeta / "entorno.json").write_text(json.dumps(info, ensure_ascii=False, indent=2), encoding="utf-8")
print(info)
for nombre in ("tiempos_recursos.csv", "entorno.json"):
    files.download(str(carpeta / nombre))